In [21]:
import os

# Mount Drive only if it's not already mounted
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive already mounted.")

Drive already mounted.


In [22]:
# Imports + device
import os, copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision import transforms, models
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [23]:
# Paths
PREP_ROOT = "/content/drive/MyDrive/datasets/sipakmed_prepared_binary"

train_dir = os.path.join(PREP_ROOT, "train")
val_dir   = os.path.join(PREP_ROOT, "val")
test_dir  = os.path.join(PREP_ROOT, "test")  # not used today

CHECKPOINT_DIR = "/content/drive/MyDrive/projects/sipakmed_demo/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [24]:
# Transforms + loaders
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = ImageFolder(train_dir, transform=train_tfms)
val_ds   = ImageFolder(val_dir, transform=val_tfms)

pin = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=pin)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=pin)

train_ds.classes, train_ds.class_to_idx, len(train_ds), len(val_ds)

(['negative', 'positive'], {'negative': 0, 'positive': 1}, 2834, 607)

In [26]:
# Model: Pretrained ResNet-18 → 1 logit head
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
in_features = resnet.fc.in_features
resnet.fc = nn.Linear(in_features, 1)   # 1 logit for BCEWithLogitsLoss

model = resnet.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# TRAINING


In [27]:
# Train/eval functions
def train_one_epoch(model, loader):
    model.train()
    running_loss = 0.0

    for x, y in tqdm(loader, desc="train", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True).float()  # BCE needs float

        logits = model(x).squeeze(1)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

@torch.no_grad()
def eval_auc(model, loader):
    model.eval()
    all_probs = []
    all_targets = []

    for x, y in tqdm(loader, desc="val", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True).float()

        logits = model(x).squeeze(1)
        probs = torch.sigmoid(logits)

        all_probs.append(probs.detach().cpu().numpy())
        all_targets.append(y.detach().cpu().numpy())

    probs = np.concatenate(all_probs)
    targets = np.concatenate(all_targets)
    auc = roc_auc_score(targets, probs)
    return auc

In [28]:
#Training loop
best_auc = -1.0
best_path = os.path.join(CHECKPOINT_DIR, "best_resnet18_bce.pth")

epochs = 5
for epoch in range(1, epochs+1):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader)
    val_auc = eval_auc(model, val_loader)

    improved = val_auc > best_auc
    if improved:
        best_auc = val_auc
        torch.save({
            "model_state": model.state_dict(),
            "class_to_idx": train_ds.class_to_idx,
            "epoch": epoch,
            "val_auc": val_auc,
        }, best_path)

    print(f"Epoch {epoch}/{epochs} | loss={train_loss:.4f} | val_auc={val_auc:.4f} | best={best_auc:.4f} | saved={improved} | time={(time.time()-t0):.1f}s")

print("Best checkpoint:", best_path)


Epoch 1/5 | loss=0.1822 | val_auc=0.9953 | best=0.9953 | saved=True | time=342.3s


Epoch 2/5 | loss=0.0924 | val_auc=0.9975 | best=0.9975 | saved=True | time=21.4s


Epoch 3/5 | loss=0.0612 | val_auc=0.9988 | best=0.9988 | saved=True | time=22.3s


Epoch 4/5 | loss=0.0519 | val_auc=0.9988 | best=0.9988 | saved=False | time=22.0s


Epoch 5/5 | loss=0.0384 | val_auc=0.9996 | best=0.9996 | saved=True | time=21.0s
Best checkpoint: /content/drive/MyDrive/projects/sipakmed_demo/checkpoints/best_resnet18_bce.pth


# TESTING AND EVALUATION

In [31]:
# CREATING TEST LOADER
# Test loader (uses val_tfms: no augmentation)
test_ds = ImageFolder(test_dir, transform=val_tfms)
test_loader = DataLoader(
    test_ds,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("classes:", test_ds.classes)
print("class_to_idx:", test_ds.class_to_idx)
print("n_test:", len(test_ds))

classes: ['negative', 'positive']
class_to_idx: {'negative': 0, 'positive': 1}
n_test: 608


In [34]:
#  Load best checkpoint
import torch

ckpt = torch.load(best_path, map_location=device, weights_only=False)  # trusted checkpoint
model.load_state_dict(ckpt["model_state"])
model.eval()

print("Loaded:", best_path)
print("ckpt epoch:", ckpt.get("epoch"), "val_auc:", ckpt.get("val_auc"))

Loaded: /content/drive/MyDrive/projects/sipakmed_demo/checkpoints/best_resnet18_bce.pth
ckpt epoch: 5 val_auc: 0.9995608404837512


In [35]:
#Predict on test + compute metrics
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)

@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    all_probs, all_targets = [], []
    for x, y in tqdm(loader, desc="test", leave=False):
        x = x.to(device, non_blocking=True)
        logits = model(x).squeeze(1)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.append(probs)
        all_targets.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_targets)

probs, targets = predict_probs(model, test_loader)

test_auc = roc_auc_score(targets, probs)
print("Test AUC:", round(test_auc, 6))

thr = 0.5
preds = (probs >= thr).astype(int)

cm = confusion_matrix(targets, preds)  # [[TN, FP],[FN, TP]]
tn, fp, fn, tp = cm.ravel()

acc  = accuracy_score(targets, preds)
prec = precision_score(targets, preds, zero_division=0)
rec  = recall_score(targets, preds, zero_division=0)   # sensitivity
spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
f1   = f1_score(targets, preds, zero_division=0)

print("Confusion matrix:\n", cm)
print(f"acc={acc:.4f}  precision={prec:.4f}  sensitivity/recall={rec:.4f}  specificity={spec:.4f}  f1={f1:.4f}")

Test AUC: 0.999225
Confusion matrix:
 [[358   4]
 [  3 243]]
acc=0.9885  precision=0.9838  sensitivity/recall=0.9878  specificity=0.9890  f1=0.9858


In [36]:
# (Quick sanity check) No file overlap between splits
import os

def basenames(ds):
    return set(os.path.basename(p) for p, _ in ds.samples)

train_names = basenames(train_ds)
val_names   = basenames(val_ds)
test_names  = basenames(test_ds)

print("train∩val :", len(train_names & val_names))
print("train∩test:", len(train_names & test_names))
print("val∩test  :", len(val_names & test_names))

train∩val : 0
train∩test: 0
val∩test  : 0


In [37]:
# Find best threshold by Youden's J (TPR - FPR)
from sklearn.metrics import roc_curve

fpr, tpr, thr = roc_curve(targets, probs)
j = tpr - fpr
best_i = j.argmax()
best_thr = thr[best_i]

print("Best threshold (Youden J):", float(best_thr))
print("TPR at best thr:", float(tpr[best_i]))
print("FPR at best thr:", float(fpr[best_i]))

Best threshold (Youden J): 0.5687777400016785
TPR at best thr: 0.9878048780487805
FPR at best thr: 0.0055248618784530384


In [38]:
# To compute confusion matrix + metrics at threshold = 0.5688
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

best_thr = 0.5687777400016785

preds = (probs >= best_thr).astype(int)
cm = confusion_matrix(targets, preds)  # [[TN, FP],[FN, TP]]
tn, fp, fn, tp = cm.ravel()

acc  = accuracy_score(targets, preds)
prec = precision_score(targets, preds, zero_division=0)
rec  = recall_score(targets, preds, zero_division=0)   # sensitivity
spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
f1   = f1_score(targets, preds, zero_division=0)

print("Threshold:", best_thr)
print("Confusion matrix:\n", cm)
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"acc={acc:.4f}  precision={prec:.4f}  sensitivity={rec:.4f}  specificity={spec:.4f}  f1={f1:.4f}")

Threshold: 0.5687777400016785
Confusion matrix:
 [[360   2]
 [  3 243]]
TN=360, FP=2, FN=3, TP=243
acc=0.9918  precision=0.9918  sensitivity=0.9878  specificity=0.9945  f1=0.9898
